# Field Boundary Detection in WorldView Imagery (Bangladesh): RGB versus RGB + NIR

## Overview

Smallholder fields in coastal Bangladesh are only a few tenths of a hectare and are separated by paddy bunds that are often narrower than one image pixel. This notebook follows the protocol of **Yang, Ahmed et al. (2020), *Remote Sensing Applications: Society and Environment* 20, 100413**, who detected functional field boundaries in WorldView imagery with fully convolutional networks, and repeats it step by step with the GeoLattice edge-detection tools:

1. read an eight-band WorldView-2 scene and the digitised field polygons;
2. turn the polygons into a two-class reference (class 1 = **boundary**, class 2 = **non-boundary** field interior);
3. cut the scene into $192 \times 192$ tiles, **split before augmenting**, and augment the training tiles with 90°, 180° and 270° rotations;
4. compare **3 channels (blue, green, red)** with **4 channels (blue, green, red + NIR2)** for every method;
5. score traditional edge detectors (Roberts, Sobel, Laplacian of Gaussian, Canny), a random forest on texture features, and the GeoLattice networks (`FC-DenseNet56/67/103`, U-Net) with the paper's metrics: precision, recall, F1 (equation 3 of the paper), ROC and AUC, training/validation log-loss and training time.

> **Important differences from the paper.** The paper used a 1.24 m WorldView-3 scene with 51 base tiles of $192 \times 192$ pixels. The scene shipped with GeoLattice is a **1.6 m WorldView-2** subset (702 × 760 pixels) with **567 labelled parcels**, so it yields only 13 usable base tiles. Absolute scores here are therefore *not* comparable with the paper, and the RGB-versus-NIR comparison is a small-sample experiment. Results are reported with all the caveats in Section 9.

## Learning objectives

- Build a boundary/non-boundary reference from field polygons and mask the unlabelled area.
- Split a small scene **spatially** (and before augmentation) so that no tile overlaps another split.
- Compare classical gradient/ridge detectors with learned detectors using pixel-wise precision, recall, F1, ROC/AUC and a 1-pixel tolerance score.
- Quantify the benefit of the NIR band against the cost in parameters and training time.
- Convert the best boundary-probability map into georeferenced field polygons.

## Contents

1. Environment and data audit
2. Building the boundary reference
3. Tiling, spatial split and rotation augmentation
4. Evaluation protocol
5. Traditional edge detectors (3 and 4 channels)
6. Random forest on texture features
7. Fully convolutional networks (U-Net, FC-DenseNet)
8. Final comparison, instance scores and a georeferenced product
9. Limitations
10. Summary

## 1. Environment and data audit

`multi.tif` has eight WorldView-2 bands in the order coastal, blue, green, yellow, red, red edge, NIR1, NIR2 (B1–B8). `aoi_trianing.shp` (spelling as shipped) holds the digitised parcels with a `Class` attribute (*Crop* or *Fallow*) and carries **no CRS**; it is assigned the raster CRS after checking that the bounds overlap.

**Installation.** This notebook needs installation option 6 (every GeoLattice module): `pip install "geolattice[all]"`. See "Installation options" in the GeoLattice README.

In [ ]:
import glob, math, os, random, sys, tempfile, time, warnings, pickle, json
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
import geopandas as gpd
from rasterio import features
from scipy import ndimage as ndi
from skimage import filters as skf, feature as skfeat
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_fscore_support

from geolattice.rsense import edge_detectors as ed

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10})
SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

here = Path.cwd().resolve()
roots = [here, *here.parents]
DATA = next((r / sub for r in roots for sub in
             ("../../../../Data/08_remote_sensing/08_09_image_classification/Edge_Detection/BD_WorldView",
              "Data/08_remote_sensing/08_09_image_classification/Edge_Detection/BD_WorldView")
             if (r / sub).is_dir()), None)
if DATA is None:
    raise FileNotFoundError("Run Jupyter from this notebook's folder or from the GeoLattice repository root.")
CACHE = Path(tempfile.gettempdir()) / "geolattice_edge_bd_worldview"
CACHE.mkdir(exist_ok=True)

print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
for p in ("rasterio", "geopandas", "scikit-image", "scikit-learn"):
    print(f"  {p:<14}{version(p)}")
print("data :", DATA)
print("files:", sorted(f.name for f in DATA.iterdir()))


In [ ]:
with rasterio.open(DATA / "multi.tif") as src:
    RAW = src.read(masked=True)
    TRANSFORM, CRS, RES = src.transform, src.crs, src.res[0]
    BOUNDS = src.bounds
BAND_NAMES = ["B1 coastal", "B2 blue", "B3 green", "B4 yellow", "B5 red", "B6 red edge", "B7 NIR1", "B8 NIR2"]
X8 = np.stack([RAW[i].filled(np.nan) for i in range(8)]).astype("float32")
H0, W0 = X8.shape[1:]

# B1 contains saturated fill (65535) over ~39 % of the scene; every other band has none
fill = (X8 == 65535)
audit = pd.DataFrame({
    "band": BAND_NAMES,
    "masked nodata %": [100 * np.isnan(X8[i]).mean() for i in range(8)],
    "fill 65535 %": [100 * np.nanmean(fill[i]) for i in range(8)],
    "p2": [np.nanpercentile(X8[i][~fill[i]], 2) for i in range(8)],
    "median": [np.nanmedian(X8[i][~fill[i]]) for i in range(8)],
    "p98": [np.nanpercentile(X8[i][~fill[i]], 98) for i in range(8)]}).set_index("band")
print(f"scene {W0} x {H0} px, {RES:g} m pixels, {CRS}, {BOUNDS.right - BOUNDS.left:.0f} x {BOUNDS.top - BOUNDS.bottom:.0f} m")
display(audit.round(1))

# the bands used in the paper's comparison: blue, green, red, NIR2 (B1 is dropped because of its fill values)
BANDS3 = [4, 2, 1]        # red, green, blue indices into X8 (B5, B3, B2)
BANDS4 = [4, 2, 1, 7]     # + NIR2 (B8)
VALID = np.isfinite(X8[1:]).all(0) & ~(X8[1:] == 65535).any(0)
print(f"valid pixels in B2-B8: {VALID.mean() * 100:.1f} %")

gdf = gpd.read_file(DATA / "aoi_trianing.shp")
print("shapefile CRS:", gdf.crs, "| polygons:", len(gdf), gdf.Class.value_counts().to_dict())
bx = gdf.total_bounds
assert bx[0] > BOUNDS.left - 50 and bx[2] < BOUNDS.right + 50 and bx[1] > BOUNDS.bottom - 50 and bx[3] < BOUNDS.top + 50, "polygons do not overlap the raster"
gdf = gdf.set_crs(CRS)
gdf["area_m2"] = gdf.area
display(gdf.groupby("Class").area_m2.describe().round(0))

In [ ]:
def stretch(a, lo=2, hi=98):
    v = np.isfinite(a)
    l, h = np.nanpercentile(a[v], [lo, hi])
    return np.clip((a - l) / (h - l), 0, 1)

rgb = np.dstack([stretch(X8[i]) for i in BANDS3])
nir = stretch(X8[7])
fig, ax = plt.subplots(1, 3, figsize=(21, 7))
ax[0].imshow(np.nan_to_num(rgb)); ax[0].set_title("True colour (B5, B3, B2)")
ax[1].imshow(np.nan_to_num(nir), cmap="gray"); ax[1].set_title("NIR2 (B8)")
gdf.boundary.plot(ax=ax[2], color="crimson", linewidth=0.6)
ax[2].set_title(f"{len(gdf)} digitised parcels (CRS assigned from raster)"); ax[2].set_aspect("equal")
for a in ax[:2]: a.axis("off")
plt.tight_layout(); plt.show()

## 2. Building the boundary reference

The paper digitised field outlines as lines, buffered them by 0.5 m, rasterised the buffer as **class 1 (boundary)** and rasterised the fields as **class 2 (non-boundary)**. Here the outlines come from the parcel polygons:

- **Boundary**: polygon outlines rasterised with `all_touched=True`, which gives lines about one 1.6 m pixel wide (the paper's 0.5 m buffer is below one pixel at this resolution).
- **Labelled domain**: only about half of the scene is covered by parcels. Pixels farther than 2 pixels (3.2 m) from any parcel are **unlabelled** (roads, ponds, homesteads, unmapped fields) and are *ignored* in training and in every metric, so that a detector is not penalised for a legitimate edge the reference does not contain.
- **Non-boundary**: labelled-domain pixels that are not boundary.

In [ ]:
poly_mask = features.rasterize([(g, 1) for g in gdf.geometry], out_shape=(H0, W0), transform=TRANSFORM, all_touched=False).astype(bool)
line_mask = features.rasterize([(g.boundary, 1) for g in gdf.geometry], out_shape=(H0, W0), transform=TRANSFORM, all_touched=True).astype(bool)
inst_ids = features.rasterize([(g, i + 1) for i, g in enumerate(gdf.geometry)], out_shape=(H0, W0), transform=TRANSFORM, all_touched=False).astype("int32")

DOMAIN0 = ndi.binary_dilation(poly_mask, iterations=2) & VALID
BOUND0 = line_mask & DOMAIN0

# pad to 768 x 768 (four 192-pixel tiles per side); padding is invalid/unlabelled
PAD = 768
def pad(a, v=0):
    out = np.full((a.shape[:-2] + (PAD, PAD)), v, dtype=a.dtype)
    out[..., :H0, :W0] = a
    return out

IMG = pad(X8, np.nan)
VALID_P, DOMAIN, BOUND, INST = pad(VALID), pad(DOMAIN0), pad(BOUND0), pad(inst_ids)
print(f"labelled domain: {DOMAIN0.mean() * 100:.1f} % of the scene | boundary pixels: {BOUND0.sum():,} "
      f"({BOUND0.sum() / DOMAIN0.sum() * 100:.1f} % of the labelled domain)")

fig, ax = plt.subplots(1, 3, figsize=(21, 7))
ov = np.nan_to_num(rgb).copy()
ov[DOMAIN0 & ~BOUND0] = 0.6 * ov[DOMAIN0 & ~BOUND0] + 0.4 * np.array([0.2, 0.5, 1.0])
ov[BOUND0] = (1, 0.1, 0.1)
ax[0].imshow(ov); ax[0].set_title("Reference: boundary (red), non-boundary (blue), unlabelled (unchanged)")
z = (slice(150, 330), slice(150, 330))
ax[1].imshow(np.nan_to_num(rgb)[z]); ax[1].set_title("Zoom: image (180 x 180 px = 288 m)")
ax[2].imshow(ov[z]); ax[2].set_title("Zoom: reference")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 3. Tiling, spatial split and rotation augmentation

The paper cut both the image and the label raster into $192 \times 192$ tiles, **split them into training, validation and test sets first**, and only then applied 90°, 180° and 270° rotations (172 training, 8 validation and 24 test images after augmentation). The same order is used here. Because the scene is small, tiles are *non-overlapping* and are assigned to the splits at random, so no pixel appears in two splits and rotations of a test tile never reach the training set. Tiles with less than 30 % labelled pixels carry almost no supervision and are dropped.

Inputs are scaled per band with the 2nd and 98th percentiles of the **training tiles only**. Pixels outside the valid mask are set to 0 after scaling and carry the *ignore* label.

In [ ]:
TILE = 192
rows = []
for i in range(PAD // TILE):
    for j in range(PAD // TILE):
        s = (slice(i * TILE, (i + 1) * TILE), slice(j * TILE, (j + 1) * TILE))
        rows.append(dict(tile=f"r{i}c{j}", i=i, j=j, valid=VALID_P[s].mean(), labelled=DOMAIN[s].mean(), boundary_px=int(BOUND[s].sum())))
TILEDF = pd.DataFrame(rows).set_index("tile")
TILEDF["eligible"] = TILEDF.labelled >= 0.30

rng = np.random.default_rng(SEED)
elig = rng.permutation(TILEDF.index[TILEDF.eligible].to_numpy())
SPLIT = {"Test": sorted(elig[:3]), "Valid": sorted(elig[3:5]), "Train": sorted(elig[5:])}
TILEDF["split"] = "dropped"
for k, v in SPLIT.items():
    TILEDF.loc[v, "split"] = k
print({k: len(v) for k, v in SPLIT.items()}, "| dropped:", TILEDF.index[~TILEDF.eligible].tolist())
display(TILEDF.groupby("split").agg(tiles=("valid", "size"), labelled=("labelled", "mean"), boundary_px=("boundary_px", "sum")).round(2))

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(np.nan_to_num(np.dstack([pad(rgb[..., c]) for c in range(3)])))
colors = {"Train": "royalblue", "Valid": "gold", "Test": "crimson", "dropped": "white"}
for t, r in TILEDF.iterrows():
    ax.add_patch(plt.Rectangle((r.j * TILE, r.i * TILE), TILE, TILE, fill=False, ec=colors[r.split], lw=3 if r.split != "dropped" else 1,
                               ls="-" if r.split != "dropped" else ":"))
    ax.text(r.j * TILE + 8, r.i * TILE + 24, f"{t}\n{r.split}", color=colors[r.split], fontsize=9, weight="bold")
ax.set_title("Spatial split of the 192-px tiles (no overlap between splits)"); ax.axis("off")
plt.show()

In [ ]:
sl = lambda t: (slice(TILEDF.loc[t, "i"] * TILE, (TILEDF.loc[t, "i"] + 1) * TILE), slice(TILEDF.loc[t, "j"] * TILE, (TILEDF.loc[t, "j"] + 1) * TILE))

# percentile scaling fitted on training tiles only
tr_valid = np.zeros((PAD, PAD), bool)
for t in SPLIT["Train"]:
    tr_valid[sl(t)] = VALID_P[sl(t)]
LO = np.array([np.percentile(IMG[b][tr_valid], 2) for b in range(8)], dtype="float32")
HI = np.array([np.percentile(IMG[b][tr_valid], 98) for b in range(8)], dtype="float32")
NORM = np.clip((np.nan_to_num(IMG) - LO[:, None, None]) / (HI - LO)[:, None, None], 0, 1)
NORM = NORM * VALID_P[None]

LAB = np.full((PAD, PAD), 255, dtype="uint8")          # 255 = ignore (unlabelled or invalid)
LAB[DOMAIN] = 0                                         # 0 = non-boundary (paper class 2)
LAB[BOUND] = 1                                          # 1 = boundary     (paper class 1)

def tile_arrays(t, bands):
    return NORM[bands][(slice(None),) + sl(t)].copy(), LAB[sl(t)].copy()

def rotate(x, y, k):
    return np.ascontiguousarray(np.rot90(x, k, axes=(-2, -1))), np.ascontiguousarray(np.rot90(y, k, axes=(-2, -1)))

def build_set(tiles, bands, augment):
    X, Y, names = [], [], []
    for t in tiles:
        x, y = tile_arrays(t, bands)
        for k in ((0, 1, 2, 3) if augment else (0,)):
            xr, yr = rotate(x, y, k)
            X.append(xr); Y.append(yr); names.append(f"{t}_rot{90 * k}")
    return np.stack(X), np.stack(Y), names

DATASETS = {}
for tag, bands in (("RGB", BANDS3), ("RGB+NIR", BANDS4)):
    DATASETS[tag] = {s: build_set(SPLIT[s], bands, augment=(s == "Train")) for s in ("Train", "Valid", "Test")}
for s in ("Train", "Valid", "Test"):
    x, y, _ = DATASETS["RGB+NIR"][s]
    lab = y != 255
    print(f"{s:<6} images: {len(x):>3} (tensor {x.shape[1:]})  boundary share of labelled pixels: {100 * (y[lab] == 1).mean():.1f} %")

x, y, names = DATASETS["RGB+NIR"]["Train"]
fig, ax = plt.subplots(2, 4, figsize=(18, 9))
for k in range(4):
    ax[0, k].imshow(x[k][[0, 1, 2]].transpose(1, 2, 0)); ax[0, k].set_title(names[k])
    show = np.where(y[k] == 255, 0.5, y[k]); ax[1, k].imshow(show, cmap="RdBu_r", vmin=0, vmax=1)
    ax[0, k].axis("off"); ax[1, k].axis("off")
fig.suptitle("Rotation augmentation of one training tile: image (top) and label (bottom; red = boundary, grey = ignore)")
plt.tight_layout(); plt.show()

## 4. Evaluation protocol

The paper scores every detector with the pixel-wise confusion matrix of the **boundary class** and reports

$$
\text{Precision}=\frac{TP}{TP+FP},\qquad \text{Recall}=\frac{TP}{TP+FN},\qquad F_1 = 2\times\frac{\text{Precision}\times\text{Recall}}{\text{Precision}+\text{Recall}},
$$

plus the ROC curve and its area (AUC). A true positive is a boundary pixel predicted as boundary, a false positive is a non-boundary pixel predicted as boundary, and a false negative is a boundary pixel predicted as non-boundary. Two details differ from a naive implementation:

- Only **labelled-domain pixels** are counted (Section 2).
- The decision threshold of each method is chosen **on the validation tiles** (maximum pixel $F_1$) and then frozen for the test tiles. AUC is threshold free.

Because the reference lines are about one pixel wide and a 1.6 m pixel cannot resolve a 0.3 m bund, a strict pixel match is harsh: a detection shifted by one pixel counts as both an error of omission and of commission. A **tolerance score** is therefore reported next to it: precision is the share of predicted boundary pixels within one pixel (8-neighbourhood) of a reference pixel, recall is the share of reference pixels within one pixel of a prediction, and $F_1^{\pm1}$ combines them.

In [ ]:
EIGHT = np.ones((3, 3), bool)
VAL_X = {k: DATASETS[k]["Valid"] for k in DATASETS}
TEST_X = {k: DATASETS[k]["Test"] for k in DATASETS}
YV, YT = DATASETS["RGB"]["Valid"][1], DATASETS["RGB"]["Test"][1]

def prf(tp, fp, fn):
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def pixel_scores(pred, Y):
    m, g = Y != 255, Y == 1
    return prf(int((pred & g & m).sum()), int((pred & ~g & m).sum()), int((~pred & g & m).sum()))

def tolerance_scores(pred, Y):
    P_hit = P_tot = R_hit = R_tot = 0
    for p, y in zip(pred, Y):
        m, g = y != 255, y == 1
        p = p & m
        near_g = ndi.binary_dilation(g, EIGHT); near_p = ndi.binary_dilation(p, EIGHT)
        P_hit += int((p & near_g).sum()); P_tot += int(p.sum())
        R_hit += int((g & near_p).sum()); R_tot += int(g.sum())
    P, R = P_hit / max(P_tot, 1), R_hit / max(R_tot, 1)
    return P, R, (2 * P * R / (P + R) if P + R else 0.0)

def thresholds(S, Y, n=60):
    v = S[Y != 255]
    return np.unique(np.quantile(v, np.linspace(0.5, 0.995, n))) if len(np.unique(v)) > 8 else np.unique(v)[1:]

def tune_threshold(S, Y):
    best = (-1, None)
    for t in thresholds(S, Y):
        f = pixel_scores(S >= t, Y)[2]
        if f > best[0]:
            best = (f, t)
    return best[1], best[0]

def evaluate_method(name, inputs, Sv, St, **extra):
    thr, f_val = tune_threshold(Sv, YV)
    pred = St >= thr
    P, R, F1 = pixel_scores(pred, YT)
    Pt, Rt, F1t = tolerance_scores(pred, YT)
    m = YT != 255
    auc = roc_auc_score((YT[m] == 1).astype(int), St[m])
    fpr, tpr, _ = roc_curve((YT[m] == 1).astype(int), St[m])
    return dict(method=name, input=inputs, threshold=float(thr), valid_F1=f_val, precision=P, recall=R, F1=F1,
                tol_precision=Pt, tol_recall=Rt, tol_F1=F1t, AUC=auc, roc=(fpr[::max(len(fpr) // 300, 1)], tpr[::max(len(tpr) // 300, 1)]), **extra)

RESULTS = []
SCORES = {}      # (method, input) -> test score maps, reused for the figures
KEYS = ["precision", "recall", "F1", "tol_F1", "AUC"]

# sanity checks of the metric code
print("reference scored against itself :", [round(v, 3) for v in pixel_scores(YT == 1, YT)])
rnd = np.random.default_rng(0).random(YT.shape)
print("random scores, tuned on valid   :", {k: round(v, 3) for k, v in evaluate_method("random", "-", np.random.default_rng(1).random(YV.shape), rnd).items() if k in KEYS})

## 5. Traditional edge detectors (3 and 4 channels)

The paper names Roberts, Sobel, Laplacian of Gaussian (LoG) and Canny as the classical operators for boundary detection. Each is applied here to every input channel and the responses are averaged, which gives the **3-channel** (red, green, blue) and **4-channel** (+ NIR2) versions:

| Detector | Edge strength used as score |
|----------|-----------------------------|
| Roberts | $\sqrt{G_x^2+G_y^2}$ from the $2\times2$ cross kernels |
| Sobel | $\sqrt{G_x^2+G_y^2}$ from the $3\times3$ kernels |
| LoG | $\lvert \nabla^2 (G_\sigma * I) \rvert$ |
| Canny | share of channels in which a pixel is a Canny edge (hysteresis thresholds from the 80th/92nd gradient percentiles) |

Every operator is applied after a Gaussian smoothing of scale $\sigma$ (Canny and LoG use $\sigma$ directly). The scale $\sigma \in \{0.7, 1, 1.5, 2, 3\}$ pixels is the only parameter and is **selected on the validation tiles**, as is the decision threshold.

In [ ]:
def edge_strength(x, kind, sigma):
    out = np.zeros(x.shape[1:], np.float32)
    for c in x:
        if kind == "Roberts":
            out += skf.roberts(ndi.gaussian_filter(c, sigma))
        elif kind == "Sobel":
            out += skf.sobel(ndi.gaussian_filter(c, sigma))
        elif kind == "LoG":
            out += np.abs(ndi.gaussian_laplace(c, sigma))
        elif kind == "Canny":
            g = ndi.gaussian_filter(c, sigma)
            gm = skf.sobel(g)
            lo, hi = np.percentile(gm, [80, 92])
            out += skfeat.canny(c, sigma=sigma, low_threshold=lo, high_threshold=hi, use_quantiles=False)
    return out / len(x)

def score_set(ds, kind, sigma):
    return np.stack([edge_strength(x, kind, sigma) for x in ds[0]])

SIGMAS = (0.7, 1.0, 1.5, 2.0, 3.0)
t0 = time.time()
sweep = []
for tag in ("RGB", "RGB+NIR"):
    for kind in ("Roberts", "Sobel", "LoG", "Canny"):
        best = None
        for s in SIGMAS:
            Sv = score_set(VAL_X[tag], kind, s)
            thr, f = tune_threshold(Sv, YV)
            sweep.append(dict(input=tag, detector=kind, sigma=s, valid_F1=f))
            if best is None or f > best[0]:
                best = (f, s)
        s = best[1]
        r = evaluate_method(kind, tag, score_set(VAL_X[tag], kind, s), score_set(TEST_X[tag], kind, s), sigma=s, train_s=0.0)
        SCORES[(kind, tag)] = score_set(TEST_X[tag], kind, s)
        RESULTS.append(r)
print(f"{time.time() - t0:.0f} s")
display(pd.DataFrame(sweep).pivot_table(index=["input", "detector"], columns="sigma", values="valid_F1").round(3)
        .style.set_caption("Validation pixel F1 for each smoothing scale (the best scale is kept)"))
display(pd.DataFrame([{k: v for k, v in r.items() if k in ["method", "input", "sigma"] + KEYS} for r in RESULTS]).round(3))

## 6. Random forest on texture features

The paper's random-forest baseline used **local binary patterns (LBP)** and **grey-level co-occurrence matrix (GLCM)** statistics (contrast, correlation, entropy and so on) with 600 trees and a maximum depth of 30. The same recipe is used here (plus a minimum leaf size of 3 and balanced class weights), with the three raw band values added so that the forest can also use colour:

- LBP (uniform, $P=8, R=1$ and $P=16, R=2$) on the luminance (mean of red, green and blue);
- GLCM statistics (contrast, energy, homogeneity, entropy, correlation) in a $7\times7$ window on the luminance quantised to 8 levels, averaged over the horizontal and vertical offsets;
- for the 4-channel version the same LBP and GLCM features are **also computed on NIR2** and the NIR value is added.

The forest is trained on all labelled pixels of the (un-augmented) training tiles; rotating a tile adds no new information to rotation-symmetric texture features. Features are computed once on the whole scene.

In [ ]:
def lbp_feats(g):
    g8 = (g * 255).astype("uint8")
    return [skfeat.local_binary_pattern(g8, 8, 1, "uniform"), skfeat.local_binary_pattern(g8, 16, 2, "uniform")]

def glcm_feats(g, levels=8, win=7):
    q = np.clip((g * levels).astype(int), 0, levels - 1)
    C = np.zeros((levels, levels) + q.shape, np.float32)
    for dy, dx in ((0, 1), (1, 0)):
        q2 = np.pad(q, ((0, dy), (0, dx)), mode="edge")[dy:, dx:]
        for a in range(levels):
            ia = (q == a)
            for b in range(levels):
                C[a, b] += ndi.uniform_filter((ia & (q2 == b)).astype(np.float32), win)
    C = (C + C.transpose(1, 0, 2, 3)) / 2
    C /= np.maximum(C.sum((0, 1), keepdims=True), 1e-9)
    a = np.arange(levels, dtype=np.float32)[:, None, None, None]; b = a.transpose(1, 0, 2, 3)
    contrast = (C * (a - b) ** 2).sum((0, 1)); energy = (C ** 2).sum((0, 1))
    homog = (C / (1 + np.abs(a - b))).sum((0, 1)); entropy = -(C * np.log(C + 1e-9)).sum((0, 1))
    mu = (C * a).sum((0, 1)); var = (C * (a - mu) ** 2).sum((0, 1))
    corr = (C * (a - mu) * (b - mu)).sum((0, 1)) / np.maximum(var, 1e-6)
    return [contrast, energy, homog, entropy, corr]

def texture_stack(g):
    return np.stack(lbp_feats(g) + glcm_feats(g)).astype("float32")

t0 = time.time()
cache = CACHE / "rf_features.npz"
if cache.exists():
    z = np.load(cache); TEX_L, TEX_N = z["L"], z["N"]
else:
    lum = NORM[[4, 2, 1]].mean(0)
    TEX_L, TEX_N = texture_stack(lum), texture_stack(NORM[7])
    np.savez(cache, L=TEX_L, N=TEX_N)
FEATS = {"RGB": np.concatenate([NORM[BANDS3], TEX_L]), "RGB+NIR": np.concatenate([NORM[BANDS4], TEX_L, TEX_N])}
print({k: v.shape[0] for k, v in FEATS.items()}, "features per pixel |", f"{time.time() - t0:.0f} s")

def tile_pixels(F, tiles):
    return np.stack([F[(slice(None),) + sl(t)] for t in tiles])

def rf_tile_scores(model, F, tiles):
    x = tile_pixels(F, tiles)                                  # (n, C, H, W)
    n, c, h, w = x.shape
    return model.predict_proba(x.transpose(0, 2, 3, 1).reshape(-1, c))[:, 1].reshape(n, h, w)

for tag in ("RGB", "RGB+NIR"):
    t0 = time.time()
    xt = tile_pixels(FEATS[tag], SPLIT["Train"]); yt = np.stack([LAB[sl(t)] for t in SPLIT["Train"]])
    keep = yt != 255
    Xtr = xt.transpose(0, 2, 3, 1)[keep]; ytr = (yt[keep] == 1).astype(int)
    rf = RandomForestClassifier(n_estimators=600, max_depth=30, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
    rf.fit(Xtr, ytr)
    dt = time.time() - t0
    Sv, St = rf_tile_scores(rf, FEATS[tag], SPLIT["Valid"]), rf_tile_scores(rf, FEATS[tag], SPLIT["Test"])
    SCORES[("Random forest", tag)] = St
    RESULTS.append(evaluate_method("Random forest", tag, Sv, St, train_s=dt, params=sum(e.tree_.node_count for e in rf.estimators_)))
    print(f"{tag:<8} RF trained on {len(ytr):,} pixels in {dt:.0f} s")
display(pd.DataFrame([{k: v for k, v in r.items() if k in ["method", "input"] + KEYS + ["train_s"]} for r in RESULTS if r["method"] == "Random forest"]).round(3))

## 7. Fully convolutional networks (U-Net, FC-DenseNet)

### 7.1 Architectures

The paper compared U-Net, SegNet and three FC-DenseNet (Tiramisu) variants. GeoLattice provides the U-Net (`unet_basic`, a plain-PyTorch encoder-decoder with skip connections) and the **checkpoint-compatible FC-DenseNet56/67/103** of the FieldEdge release through `edge_detectors.build_edge_model`. SegNet is not part of GeoLattice and FCN-DKConv is not used.

A dense block of $L$ layers with growth rate $k$ concatenates all earlier feature maps, so layer $\ell$ receives

$$
x_\ell = H_\ell\big([x_0, x_1, \ldots, x_{\ell-1}]\big),\qquad H_\ell = \text{BN}\rightarrow\text{ReLU}\rightarrow\text{Conv}_{3\times3}\rightarrow\text{Dropout},
$$

and adds $k$ channels. FC-DenseNet alternates dense blocks with transition-down ($1\times1$ convolution and max-pooling) paths and mirrors them with transition-up (transposed convolution) paths that are concatenated with the skip connections. The three variants differ in depth and growth rate:

| Model | Pooling steps | Growth rate $k$ | Layers per block |
|-------|---------------|-----------------|------------------|
| FC-DenseNet56 | 5 | 12 | 4 in each of the 11 blocks |
| FC-DenseNet67 | 5 | 16 | 5 in each of the 11 blocks |
| FC-DenseNet103 | 5 | 16 | 4, 5, 7, 10, 12 (down), 15 (bottleneck), 12, 10, 7, 5, 4 (up) |

The **4-channel version** differs from the 3-channel network only in the first $3\times3$ convolution (`in_channels=4`), so the extra cost of the NIR band is a few hundred parameters; the training-time difference is what the paper reports and is measured below.

### 7.2 Loss and training

The final layer outputs two logits per pixel (class 1 = boundary, class 2 = non-boundary). Training minimises the **class-weighted cross-entropy** (the log-loss curves of the paper)

$$
\mathcal{L} = -\frac{1}{|\Omega|}\sum_{p\in\Omega} w_{y_p}\,\log \operatorname{softmax}(z_p)_{y_p},
$$

where $\Omega$ is the set of **labelled** pixels of a batch (unlabelled pixels carry the ignore label) and $w_c \propto 1/\text{frequency}_c$ compensates for the 12 % boundary share, which the paper did not need to do explicitly. The optimiser is Adam; the model with the lowest **validation** loss is kept. Weights, dropout (0.2) and architecture are otherwise the GeoLattice defaults.

### 7.3 Hyper-parameter sensitivity

Table 6 of the paper varied learning rate, batch size, image size, kernel size and dropout. The same idea is applied to **learning rate and batch size** for FC-DenseNet56 on RGB (a fixed budget of 700 optimisation steps), and the best validation configuration is reused for all networks and both inputs.

In [ ]:
def class_weights(Y):
    lab = Y[Y != 255]
    freq = np.array([(lab == 0).mean(), (lab == 1).mean()])
    w = 1 / freq
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=DEVICE)

def to_tensor(x, y):
    return torch.from_numpy(x).float().to(DEVICE), torch.from_numpy(y.astype("int64")).to(DEVICE)

@torch.no_grad()
def predict_prob(model, x, bs=4):
    model.eval()
    out = []
    for i in range(0, len(x), bs):
        xb = torch.from_numpy(x[i:i + bs]).float().to(DEVICE)
        with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
            lg = ed._model_logits(model, xb)
        out.append(F.softmax(lg.float(), 1)[:, 1].cpu().numpy())
    return np.concatenate(out)

def train_model(arch, tag, seed=SEED, lr=3e-4, batch=4, steps=700, wd=0.0, eval_every=35):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr, Ytr, _ = DATASETS[tag]["Train"]; Xv, Yv, _ = DATASETS[tag]["Valid"]
    C = Xtr.shape[1]
    model = ed.build_edge_model(arch, in_channels=C, device=DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    cw = class_weights(Ytr)
    xt, yt = to_tensor(Xtr, Ytr); xv, yv = to_tensor(Xv, Yv)
    hist, best, best_state, run = [], 1e9, None, []
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for step in range(1, steps + 1):
        model.train()
        idx = torch.randint(0, len(xt), (batch,), device=DEVICE)
        with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
            lg = ed._model_logits(model, xt[idx])
        loss = F.cross_entropy(lg.float(), yt[idx], weight=cw, ignore_index=255)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        run.append(loss.item())
        if step % eval_every == 0:
            model.eval()
            with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
                vl = F.cross_entropy(ed._model_logits(model, xv).float(), yv, weight=cw, ignore_index=255).item()
            hist.append(dict(step=step, train_loss=float(np.mean(run)), val_loss=vl)); run = []
            if vl < best:
                best, best_state = vl, {k: v.detach().clone() for k, v in model.state_dict().items()}
    if DEVICE == "cuda": torch.cuda.synchronize()
    dt = time.time() - t0
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist), dt

def model_scores(model, tag):
    return predict_prob(model, DATASETS[tag]["Valid"][0]), predict_prob(model, DATASETS[tag]["Test"][0])

n_params = lambda m: sum(p.numel() for p in m.parameters())
t0 = time.time()
m, h, dt = train_model("FC-DenseNet56", "RGB", steps=70)
print(f"smoke test: 70 steps in {dt:.1f} s, {n_params(m) / 1e6:.2f} M parameters, final train/val loss {h.train_loss.iloc[-1]:.3f}/{h.val_loss.iloc[-1]:.3f}")

In [ ]:
sens_file = CACHE / "sensitivity.pkl"
if sens_file.exists():
    SENS = pickle.loads(sens_file.read_bytes())
else:
    SENS = []
    for lr in (1e-3, 3e-4, 1e-4):
        for batch in (2, 8):
            m, h, dt = train_model("FC-DenseNet56", "RGB", lr=lr, batch=batch, steps=700)
            Sv, _ = model_scores(m, "RGB")
            SENS.append(dict(lr=lr, batch=batch, best_val_loss=h.val_loss.min(), valid_F1=tune_threshold(Sv, YV)[1],
                             valid_AUC=roc_auc_score((YV[YV != 255] == 1).astype(int), Sv[YV != 255]), train_s=dt))
    sens_file.write_bytes(pickle.dumps(SENS))
SENS = pd.DataFrame(SENS)
display(SENS.round(4).style.highlight_max(subset=["valid_F1"], color="#cfe8cf").set_caption("Hyper-parameter sensitivity, FC-DenseNet56 on RGB (validation tiles)"))
BEST_HP = SENS.sort_values("valid_F1", ascending=False).iloc[0]
LR, BATCH = float(BEST_HP.lr), int(BEST_HP.batch)
print(f"selected: learning rate {LR:g}, batch size {BATCH}")

### 7.4 Training the four networks on both inputs

Each network is trained from scratch on the 28 augmented training tiles for 1000 optimisation steps with the learning rate and batch size selected above, once for **RGB** and once for **RGB + NIR**, with **three random seeds** (initialisation, batch order and dropout). The paper trained one model per configuration; with only 12 labelled tiles a single run cannot separate a real gain from seed noise, so the spread over seeds is reported. Training time is the wall-clock time of the optimisation loop on the GPU, including the periodic validation passes.

In [ ]:
ARCHS = {"U-Net": "unet_basic", "FC-DenseNet56": "FC-DenseNet56", "FC-DenseNet67": "FC-DenseNet67", "FC-DenseNet103": "FC-DenseNet103"}
SEEDS = (SEED, SEED + 1, SEED + 2)
STEPS = 1000
RUNS = {}
t_all = time.time()
for name, arch in ARCHS.items():
    for tag in ("RGB", "RGB+NIR"):
        for seed in SEEDS:
            f = CACHE / f"cnn_{name}_{tag}_{seed}.pkl"
            if f.exists():
                RUNS[(name, tag, seed)] = pickle.loads(f.read_bytes()); continue
            m, h, dt = train_model(arch, tag, seed=seed, lr=LR, batch=BATCH, steps=STEPS)
            Sv, St = model_scores(m, tag)
            run = dict(hist=h, train_s=dt, params=n_params(m), Sv=Sv, St=St, best_step=int(h.step[h.val_loss.idxmin()]))
            torch.save(m.state_dict(), CACHE / f"cnn_{name}_{tag}_{seed}.pt")
            f.write_bytes(pickle.dumps(run))
            RUNS[(name, tag, seed)] = run
            print(f"{name:<15}{tag:<8} seed {seed}: {dt:5.0f} s, best step {run['best_step']}", flush=True)
            del m; torch.cuda.empty_cache()
print(f"done in {time.time() - t_all:.0f} s")

CNN = []
for (name, tag, seed), r in RUNS.items():
    CNN.append(evaluate_method(name, tag, r["Sv"], r["St"], seed=seed, train_s=r["train_s"], params=r["params"], best_step=r["best_step"]))
CNNDF = pd.DataFrame([{k: v for k, v in r.items() if k != "roc"} for r in CNN])
summary = (CNNDF.groupby(["method", "input"], sort=False)
           .agg(params_M=("params", lambda s: s.iloc[0] / 1e6), train_s=("train_s", "mean"),
                precision=("precision", "mean"), recall=("recall", "mean"), F1=("F1", "mean"), F1_sd=("F1", "std"),
                tol_F1=("tol_F1", "mean"), AUC=("AUC", "mean"), AUC_sd=("AUC", "std")))
display(summary.round(3).style.set_caption("Test tiles: mean over three seeds (pixel metrics at the validation-tuned threshold)"))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.6), sharey=True)
for ax, name in zip(axes, ARCHS):
    for tag, c in (("RGB", "tab:blue"), ("RGB+NIR", "tab:red")):
        h = RUNS[(name, tag, SEEDS[0])]["hist"]
        ax.plot(h.step, h.train_loss, c=c, lw=1.5, label=f"{tag} train")
        ax.plot(h.step, h.val_loss, c=c, lw=1.5, ls="--", label=f"{tag} validation")
    ax.set_title(name); ax.set_xlabel("optimisation step"); ax.grid(alpha=0.3)
axes[0].set_ylabel("weighted log-loss"); axes[0].legend()
fig.suptitle("Training and validation log-loss (seed 2026)")
plt.tight_layout(); plt.show()

## 8. Final comparison, instance scores and a georeferenced product

In [ ]:
ALL = pd.DataFrame([{k: v for k, v in r.items() if k not in ("roc",)} for r in RESULTS] + [{k: v for k, v in r.items() if k != "roc"} for r in CNN])
ALL["seed"] = ALL.seed.fillna(0)
table = (ALL.groupby(["method", "input"], sort=False)
         .agg(precision=("precision", "mean"), recall=("recall", "mean"), F1=("F1", "mean"), F1_sd=("F1", "std"),
              tol_F1=("tol_F1", "mean"), AUC=("AUC", "mean"), AUC_sd=("AUC", "std"), train_s=("train_s", "mean")))
order = ["Roberts", "Sobel", "LoG", "Canny", "Random forest", "U-Net", "FC-DenseNet56", "FC-DenseNet67", "FC-DenseNet103"]
table = table.reindex(pd.MultiIndex.from_product([order, ["RGB", "RGB+NIR"]], names=["method", "input"]))
display(table.round(3).style.format(precision=3, na_rep="").background_gradient(subset=["F1", "AUC"], cmap="Greens").set_caption("Test tiles: all methods"))

# paired RGB -> RGB+NIR differences (CNNs: per seed; others: one run)
d = ALL.pivot_table(index=["method", "seed"], columns="input", values=["F1", "AUC", "tol_F1"])
gain = pd.DataFrame({m: {"dF1": (d.loc[m]["F1"]["RGB+NIR"] - d.loc[m]["F1"]["RGB"]).mean(),
                         "dF1_sd": (d.loc[m]["F1"]["RGB+NIR"] - d.loc[m]["F1"]["RGB"]).std(),
                         "dAUC": (d.loc[m]["AUC"]["RGB+NIR"] - d.loc[m]["AUC"]["RGB"]).mean(),
                         "dtol_F1": (d.loc[m]["tol_F1"]["RGB+NIR"] - d.loc[m]["tol_F1"]["RGB"]).mean(),
                         "seeds_with_gain": int(((d.loc[m]["F1"]["RGB+NIR"] - d.loc[m]["F1"]["RGB"]) > 0).sum()),
                         "n": len(d.loc[m])} for m in order}).T
display(gain.round(3).style.set_caption("Effect of adding NIR2 (RGB+NIR minus RGB), test tiles"))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(23, 6))
for k, tag in enumerate(("RGB", "RGB+NIR")):
    for r in [*RESULTS, *[c for c in CNN if c["seed"] == SEEDS[0]]]:
        if r["input"] != tag: continue
        fpr, tpr = r["roc"]
        ax[k].plot(fpr, tpr, lw=1.6, label=f"{r['method']} (AUC {r['AUC']:.2f})")
    ax[k].plot([0, 1], [0, 1], "k:", lw=1); ax[k].set_title(f"ROC curves, test tiles, {tag}")
    ax[k].set_xlabel("false positive rate"); ax[k].set_ylabel("true positive rate"); ax[k].legend(fontsize=8, loc="lower right"); ax[k].grid(alpha=0.3)
tt = ALL.groupby(["method", "input"], sort=False).train_s.mean().unstack().reindex(order).replace(0, np.nan)
tt.dropna(how="all").plot.bar(ax=ax[2], color=["tab:blue", "tab:red"], rot=45); ax[2].set_yscale("log")
ax[2].set_ylabel("training time (s, log)"); ax[2].set_title("Training time"); ax[2].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()